# 03 — Deep Learning Models vs. Reference Baselines

### Academic Research Objectives:
1. **Comparative Benchmarking:** Evaluate multi-horizon contagion prediction across the complete model zoo:
   - **VAR-GFEVD:** Statistical econometric baseline.
   - **Independent Cascade:** Probabilistic epidemic diffusion baseline.
   - **CryptoLSTM:** Non-graph temporal recurrent baseline.
   - **CryptoTGN:** Continuous-time dynamic graph neural network with evolving memory.
   - **DyGFormer Feasibility:** Dynamic graph transformer comparison.
2. **Metrics:** Prioritize **AUPRC (Average Precision)** for node classification given severe class imbalance (<5% positive shocks), alongside AUROC, RMSE, and MAE for multi-horizon returns.
3. **Repeated Seeds & Statistical Testing:** Evaluate model stability across repeated random seeds.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from cryptograph.data.universe import TARGET_UNIVERSE_30
from cryptograph.models.tgn import CryptoTGN
from cryptograph.models.lstm import CryptoLSTM
from cryptograph.training.seed import seed_everything
from cryptograph.training.device import resolve_device

device = resolve_device("auto")
seed_everything(42)
print(f"Active Device: {device}")


C:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-10-03 00:00:34] [INFO] [cryptograph.training.device] - Using device: cpu


[2026-10-03 00:00:34] [INFO] [cryptograph.training.seed] - Initialized global deterministic seed: 42


Active Device: cpu


## 1. Load Processed 30-Coin Hourly Dataset & Chronological Splits


In [2]:
dataset_path = Path("../data/processed/v2/1h/processed_dataset.pt")
data = torch.load(dataset_path, map_location="cpu", weights_only=False)

features = data["features"]
targets_shock = data["targets_shock"]
targets_return = data["targets_return"]
validity = data["target_validity_mask"]
active_mask = data["active_mask"]
split_indices = data["split_indices"]

train_idx = np.array(split_indices["train_indices"])
val_idx = np.array(split_indices["val_indices"])
test_idx = np.array(split_indices["test_indices"])

print(f"Features: {features.shape}, Targets: {targets_shock.shape}")
print(f"Partitions: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")


Features: torch.Size([24024, 30, 9]), Targets: torch.Size([24024, 30, 5])
Partitions: Train=14616, Val=4536, Test=4536


## 2. Train and Evaluate Non-Graph Baseline (`CryptoLSTM`)


In [3]:
# Load best trained CryptoLSTM checkpoint and evaluate architecture
import glob
lstm_ckpts = sorted(glob.glob('../runs/*lstm*/best_lstm_checkpoint.pt'))
if lstm_ckpts:
    ckpt_file = lstm_ckpts[-1]
    print(f'Loading latest LSTM checkpoint: {ckpt_file}')
    ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    lstm_model = CryptoLSTM(in_features=features.shape[-1], hidden_dim=64, num_layers=2, out_horizons=len(data['horizons']))
    lstm_model.load_state_dict(ckpt['model_state_dict'])
    lstm_model.eval()
    print('CryptoLSTM successfully loaded with 2 LSTM layers and multi-horizon head.')
    print(f'Best Validation Mean AUPRC: {ckpt.get("best_val_auprc", 0.0):.4f}')
else:
    print('No saved LSTM checkpoint found.')


Loading latest LSTM checkpoint: ../runs\20261002-113205_lstm_seed_44_s44_195e9f5\best_lstm_checkpoint.pt
CryptoLSTM successfully loaded with 2 LSTM layers and multi-horizon head.
Best Validation Mean AUPRC: 0.7580


## 3. Train and Evaluate Continuous-Time Dynamic Graph Network (`CryptoTGN`)


In [4]:
# Load best trained CryptoTGN checkpoint
tgn_ckpts = sorted(glob.glob('../runs/*tgn*/best_tgn_checkpoint.pt'))
if tgn_ckpts:
    ckpt_file = tgn_ckpts[-1]
    print(f'Loading latest TGN checkpoint: {ckpt_file}')
    ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    tgn_model = CryptoTGN(num_nodes=30, raw_msg_dim=9, node_feat_dim=16, memory_dim=64, embedding_dim=64, out_horizons=5)
    tgn_model.load_state_dict(ckpt['model_state_dict'])
    tgn_model.eval()
    print('CryptoTGN successfully loaded with dynamic memory and spatial GCNConv.')
    print(f'Best Validation Mean AUPRC: {ckpt.get("best_val_auprc", 0.0):.4f}')
else:
    print('No saved TGN checkpoint found.')


Loading latest TGN checkpoint: ../runs\20261002-115105_tgn_seed_44_s44_195e9f5\best_tgn_checkpoint.pt


CryptoTGN successfully loaded with dynamic memory and spatial GCNConv.
Best Validation Mean AUPRC: 0.6715


## 4. Multi-Horizon Performance Comparison & Empirical Leaderboard


In [5]:
# Display Comparative Leaderboard from reports/model_comparison.csv
comp_file = Path('../reports/model_comparison.csv')
if comp_file.exists():
    df_comp = pd.read_csv(comp_file)
    pivot_auprc = df_comp.pivot(index='horizon_hours', columns='model', values='test_auprc')
    pivot_auroc = df_comp.pivot(index='horizon_hours', columns='model', values='test_auroc')
    
    print('=== Test AUPRC Across Horizons (Primary Imbalanced Metric) ===')
    print(pivot_auprc.to_string())
    print('\n=== Test AUROC Across Horizons ===')
    print(pivot_auroc.to_string())
    
    fig = px.line(
        df_comp,
        x='horizon_hours',
        y='test_auprc',
        color='model',
        markers=True,
        title='Contagion Shock Forecasting: Test AUPRC vs. Horizon (1h - 24h)',
        labels={'horizon_hours': 'Forecast Horizon (Hours)', 'test_auprc': 'AUPRC (Average Precision)'}
    )
    fig.update_layout(template='plotly_white')
    fig.show()
else:
    print('reports/model_comparison.csv not found.')


=== Test AUPRC Across Horizons (Primary Imbalanced Metric) ===
model          CryptoDyGFormer  CryptoLSTM  CryptoTGN  IndependentCascade
horizon_hours                                                            
1                     0.857611    0.872274   0.690241            0.003614
3                     0.845294    0.857717   0.687648            0.009921
6                     0.825312    0.834697   0.669911            0.018430
12                    0.786811    0.794222   0.639287            0.034364
24                    0.706891    0.710822   0.576559            0.060153

=== Test AUROC Across Horizons ===
model          CryptoDyGFormer  CryptoLSTM  CryptoTGN  IndependentCascade
horizon_hours                                                            
1                     0.995533    0.995757   0.974454                 0.5
3                     0.993741    0.993415   0.971392                 0.5
6                     0.989586    0.989059   0.964653                 0.5
12           